In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
# Set the database path to a location with write permissions
db_path = '../race_league_results.db'

# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()


# How to rank...
1. Determine each racer's percentile for every race
2. Get a median percentile for the year
3. Ranking is based on your last 3 years -> take median from last year, or the median before that, or 3 years past. If non exist, use the all time median. Otherwise, Null...

```py
df['_percentile'] = df.groupby('race_id')['_time'].rank(pct=True, ascending=True)
    
    # Calculating median percentile rank for each racer per year
    median_percentiles = df.groupby(['racer_id', 'year'])['_percentile'].median().reset_index()
```

In [3]:
def compute_rank(row, cols, fallback_col):
    """
    row: the row from the DataFrame
    cols: list of columns to consider (e.g. [cols[-1], cols[-2], cols[-3]])
    fallback_col: the fallback column name if all three are NaN (e.g. 'all_time_avg_percentile')
    """
    # Example weights
    w1, w2, w3 = 0.3, 0.5, 0.2
    
    values = [row[cols[0]], row[cols[1]], row[cols[2]]]
    weights = [w1, w2, w3]
    
    # Filter only non-null values and their corresponding weights
    valid_pairs = [(val, wt) for val, wt in zip(values, weights) if pd.notnull(val)]
    
    if not valid_pairs:
        # If all are null, return the fallback value
        return row[fallback_col]
    else:
        # Weighted average over the non-null columns
        total_weight = sum(pair[1] for pair in valid_pairs)
        weighted_sum = sum(pair[0] * pair[1] for pair in valid_pairs)
        return weighted_sum / total_weight

# Function to calculate aggregated and last 3 years race result percentiles for a given dataset
def calculate_aggregated_percentiles(df):
    # Calculating percentile ranks for each race
    # Lower times are better, so we use ascending=True
    df['percentile'] = df.groupby('race_id')['best_time'].rank(pct=True, ascending=True)
    df['position'] = df.groupby('race_id')['best_time'].rank(method='min', ascending=True).astype(int)
    
    # Calculating median percentile rank for each racer per year
    median_percentiles = df.groupby(['racer_id', 'year'])['percentile'].median().reset_index()
    
    # Creating a dictionary to hold pre-season and post-season rankings for each year
    rankings = {}
    
    # Iterating over each year present in the dataset
    for year in sorted(median_percentiles['year'].unique()):
        # Getting the median percentiles for the current year
        current_year_rankings = median_percentiles[median_percentiles['year'] == year]
    
        # Sorting by median percentile (better performance has lower percentile)
        current_year_rankings = current_year_rankings.sort_values('percentile')
    
        # Assigning post-season rankings based on sorted order
        current_year_rankings['post_season_rank'] = range(1, len(current_year_rankings) + 1)
    
        # Assigning pre-season rankings
        if year - 1 in rankings:
            # If the previous year exists, use its post-season rankings
            previous_year_rankings = rankings[year - 1][['racer_id', 'post_season_rank']]
            current_year_rankings = current_year_rankings.merge(previous_year_rankings, on='racer_id', how='left', suffixes=('', '_prev'))
            current_year_rankings['pre_season_rank'] = current_year_rankings['post_season_rank_prev'].fillna(max(current_year_rankings['post_season_rank']) + 1)
            current_year_rankings.drop(columns=['post_season_rank_prev'], inplace=True)
        else:
            # If no previous year data, assign a default pre-season rank
            current_year_rankings['pre_season_rank'] = max(current_year_rankings['post_season_rank']) + 1
    
        # Adding to the rankings dictionary
        rankings[year] = current_year_rankings
    
    # Combining all years' rankings into a single DataFrame
    all_years_rankings = pd.concat(rankings.values())
    
    # Calculating all-time percentile ranking for each racer: TODO, update this!
    all_time_percentiles = df.groupby('racer_id')['percentile'].mean().reset_index()
    all_time_percentiles.rename(columns={'percentile': 'all_time_avg_percentile'}, inplace=True)

    # Getting the last 3 years
    last_3_years = df['year'].unique()[-3:]
    columns = []

    # Initializing a DataFrame to hold last 3 years' percentiles
    last_3_years_percentiles = pd.DataFrame()

    # Iterating over each of the last 3 years
    for year in last_3_years:
        # Calculating median percentile for each racer for the given year
        year_percentiles = df[df['year'] == year].groupby('racer_id')['percentile'].median().reset_index()
        year_percentiles.rename(columns={'percentile': f'median_percentile_{year}'}, inplace=True)
        columns.append(f'median_percentile_{year}')

        # Merging with the main DataFrame
        if last_3_years_percentiles.empty:
            last_3_years_percentiles = year_percentiles
        else:
            last_3_years_percentiles = last_3_years_percentiles.merge(year_percentiles, on='racer_id', how='outer')

    # Merging all-time percentiles with last 3 years' percentiles
    aggregated_df = all_time_percentiles.merge(last_3_years_percentiles, on='racer_id', how='left')

    ## Finally, create a ranking from the last 3 years and avg data:    
    #aggregated_df['rank'] = aggregated_df.apply(
    #    lambda x: x[columns[-1]] if pd.notnull(x[columns[-1]]) else x[columns[-2]] if pd.notnull(x[columns[-2]]) else x[columns[-3]] if pd.notnull(x[columns[-3]]) else x['all_time_avg_percentile'],
    #    axis=1
    #)
    
    # Example usage in your apply:
    aggregated_df['rank'] = aggregated_df.apply(
        lambda x: compute_rank(
            row=x,
            cols=[columns[-1], columns[-2], columns[-3]],  # or however you define your three columns
            fallback_col='all_time_avg_percentile'
        ),
        axis=1
    )

    return aggregated_df, all_years_rankings

def last_5_races(df):
    df = df.sort_values(["racer_id", "race_date"])
    last_5 = df.groupby("racer_id", group_keys=False).tail(5)
    stats = (
        last_5.groupby("racer_id")["position"]
              .agg(avg_position_last5=("mean"), median_position_last5=("median"))
              .reset_index()
    )
    return stats

need a df with: racer_id, best_time, race_id, race_date (year?)

Make sure to filter out any race where the racer did not finish! -> i.e., best time < 9998

In [4]:
sql = """
SELECT results.race_id, results.racer_id, results.discipline, results.best_time, races.race_date, races.year
FROM (
    SELECT racer_id, discipline, best_time, race_id
    FROM RaceResults
    WHERE COALESCE(best_time, 9999) < 9998
) AS results
JOIN (
    SELECT race_id, race_date, substr(description, 1,4) as year
    FROM Races
) AS races
ON results.race_id = races.race_id
"""

df_combined = pd.read_sql_query(sql, conn)
df_combined['year'] = df_combined['year'].astype(int)

In [5]:
first_year = df_combined.groupby("racer_id")["year"].min()
df_combined["is_rockie"] = df_combined["racer_id"].map(first_year) == 2026

In [6]:
#mikemctaggart -> michaelmctaggart
df_combined.loc[df_combined['racer_id'] == 'mikemctaggart', 'racer_id'] = 'michaelmctaggart'

#williamcartar -> willcartar
df_combined.loc[df_combined['racer_id'] == 'williamcartar', 'racer_id'] = 'willcartar'

#mcleanwood -> mcleanjamiesonwood
df_combined.loc[df_combined['racer_id'] == 'mcleanwood', 'racer_id'] = 'mcleanjamiesonwood'

#stephaniecoward -> stephcoward
df_combined.loc[df_combined['racer_id'] == 'stephaniecoward', 'racer_id'] = 'stephcoward'


#nicholasbalan -> nickbalan
df_combined.loc[df_combined['racer_id'] == 'nicholasbalan', 'racer_id'] = 'nickbalan'

#jeffkansun -> jeffreykansun
df_combined.loc[df_combined['racer_id'] == 'jeffkansun', 'racer_id'] = 'jeffreykansun'

#randymilthorpe -> robertmilthorpe
df_combined.loc[df_combined['racer_id'] == 'randymilthorpe', 'racer_id'] = 'robertmilthorpe'

#mitchperreault -> mitchperrault
df_combined.loc[df_combined['racer_id'] == 'mitchperreault', 'racer_id'] = 'mitchperrault'

#joannaperreault -> joannaperrault
df_combined.loc[df_combined['racer_id'] == 'joannaperreault', 'racer_id'] = 'joannaperrault'

In [7]:
df_combined.head(2)

,race_id,racer_id,discipline,best_time,race_date,year,is_rockie
0,1,jeffcox,SKI,35.82,2014-01-01 00:00:00,2014,False
1,1,mcleanjamiesonwood,SKI,36.10,2014-01-01 00:00:00,2014,False


In [8]:
ski_df = df_combined[df_combined['discipline'] == 'SKI']
snbd_df = df_combined[df_combined['discipline'] == 'SNBD']

In [9]:
ski_aggregated_df, ski_all_time = calculate_aggregated_percentiles(ski_df)
snbd_aggregated_df, snbd_all_time = calculate_aggregated_percentiles(snbd_df)

/var/folders/cp/tk2rjzwj4n76t_7mcr1f2y640000gn/T/ipykernel_26288/2388674575.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['percentile'] = df.groupby('race_id')['best_time'].rank(pct=True, ascending=True)
/var/folders/cp/tk2rjzwj4n76t_7mcr1f2y640000gn/T/ipykernel_26288/2388674575.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['position'] = df.groupby('race_id')['best_time'].rank(method='min', ascending=True).astype(int)
/var/folders/cp/tk2rjzwj4n76t_7mcr1f2y640000gn/T/ipykernel_26288/23886

In [10]:
ski_aggregated_df.sort_values(by='rank', ascending=True, inplace=True)

In [11]:
ski_aggregated_df.head(5)

,racer_id,all_time_avg_percentile,median_percentile_2024,median_percentile_2025,median_percentile_2026,rank
61,caleweinberg,0.025832,NaN,NaN,NaN,0.025832
289,nickbalan,0.049123,0.022856,0.044118,0.012346,0.030334
305,perrywatt,0.035383,NaN,NaN,NaN,0.035383
272,michaelmctaggart,0.029651,0.027397,0.023810,0.060241,0.035457
145,gregwega,0.047619,NaN,NaN,NaN,0.047619


## Lots of folks are missing, why?

In [23]:
ski_df.head(2)

,race_id,racer_id,discipline,best_time,race_date,year,percentile,position
0,1,jeffcox,SKI,35.82,2014-01-01 00:00:00,2014,0.017544,1
1,1,mcleanwood,SKI,36.10,2014-01-01 00:00:00,2014,0.035088,2


In [17]:
#McLean
ski_df.query('racer_id.str.contains("randymilthorpe")')

,race_id,racer_id,discipline,best_time,race_date,year,is_rockie,percentile,position
100,2,randymilthorpe,SKI,29.00,2013-01-01 00:00:00,2013,False,0.413333,31
190,3,randymilthorpe,SKI,41.91,2013-01-01 00:00:00,2013,False,0.390625,25
268,4,randymilthorpe,SKI,41.89,2013-01-01 00:00:00,2013,False,0.327869,20
447,6,randymilthorpe,SKI,47.10,2013-01-01 00:00:00,2013,False,0.539683,34
541,7,randymilthorpe,SKI,45.55,2014-01-12 00:00:00,2014,False,0.454545,35
636,8,randymilthorpe,SKI,45.46,2014-01-19 00:00:00,2014,False,0.522059,35
714,9,randymilthorpe,SKI,41.93,2014-02-02 00:00:00,2014,False,0.417910,28
794,10,randymilthorpe,SKI,44.04,2014-02-22 00:00:00,2014,False,0.475410,29
879,11,randymilthorpe,SKI,32.06,2014-12-31 00:00:00,2015,False,0.583333,35
947,12,randymilthorpe,SKI,42.60,2015-01-05 00:00:00,2015,False,0.385542,32


# Join to start lists, and append last 5 race results!

In [12]:
from helper_functions import clean_string

In [13]:
start_list = pd.read_csv("2026/2026_Start_List.csv")
start_list.tail(2)

,bib,CAT,name,tier,team
119,188,SNBD,Jonathan Ages,NaN,NaN
120,213,SNBD,Graham Ramshaw,14.0,Blue


In [14]:
ski_list = start_list[start_list.CAT == 'SKI']
snbd_list = start_list[start_list.CAT == 'SNBD']

ski_list["racer_id"] = ski_list["name"].apply(clean_string)
snbd_list["racer_id"] = snbd_list["name"].apply(clean_string)

/var/folders/cp/tk2rjzwj4n76t_7mcr1f2y640000gn/T/ipykernel_26288/839224227.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ski_list["racer_id"] = ski_list["name"].apply(clean_string)
/var/folders/cp/tk2rjzwj4n76t_7mcr1f2y640000gn/T/ipykernel_26288/839224227.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snbd_list["racer_id"] = snbd_list["name"].apply(clean_string)


In [15]:
ski_df_last5 = last_5_races(ski_df)
snbd_df_last5 = last_5_races(snbd_df)

In [20]:
joined_ski = ski_list.merge(ski_aggregated_df, how="left", on="racer_id")
joined_ski = joined_ski.merge(ski_df_last5, how="left", on="racer_id")
joined_ski = joined_ski.merge(ski_df[["racer_id", "is_rockie"]].drop_duplicates(), how="left", on="racer_id")

joined_ski.sort_values(by='rank', ascending=True, inplace=True)
joined_ski

,bib,CAT,name,tier,team,racer_id,all_time_avg_percentile,median_percentile_2024,median_percentile_2025,median_percentile_2026,rank,avg_position_last5,median_position_last5,is_rockie
103,111,SKI,Nick Balan,1.0,Green Machine,nickbalan,0.049123,0.022856,0.044118,0.012346,0.030334,1.8,1.0,False
101,109,SKI,Michael McTaggart,1.0,Yellow,michaelmctaggart,0.029651,0.027397,0.023810,0.060241,0.035457,4.6,5.0,False
100,108,SKI,Rob Kansun,1.0,Big Eggplant Energy,robkansun,0.070793,NaN,0.057419,0.044221,0.052470,5.0,3.0,False
86,95,SKI,Nick Parr,3.0,Orange,nickparr,0.030682,NaN,NaN,0.060606,0.060606,3.0,3.0,False
98,106,SKI,Jordan Kofman,2.0,Red,jordankofman,0.037356,NaN,NaN,0.064584,0.064584,6.0,8.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,4,SKI,Alison Pridham,13.0,Big Eggplant Energy,alisonpridham,0.979113,NaN,NaN,0.979798,0.979798,84.4,82.0,True
5,6,SKI,Talia Laurie,13.0,Green Machine,talialaurie,0.979115,NaN,1.000000,0.962699,0.986012,79.4,81.0,False
23,27,SKI,Leah Bacon,11.0,Pink Powder Club,leahbacon,0.988852,NaN,NaN,0.988852,0.988852,89.5,89.5,True
0,1,SKI,Kenny Freeman,13.0,Yellow,kennyfreeman,1.000000,NaN,NaN,1.000000,1.000000,86.2,83.0,True


In [21]:
joined_snbd = snbd_list.merge(snbd_aggregated_df, how="left", on="racer_id")
joined_snbd = joined_snbd.merge(snbd_df_last5, how="left", on="racer_id")
joined_snbd = joined_snbd.merge(snbd_df[["racer_id", "is_rockie"]].drop_duplicates(), how="left", on="racer_id")

joined_snbd.sort_values(by='rank', ascending=True, inplace=True)
joined_snbd

,bib,CAT,name,tier,team,racer_id,all_time_avg_percentile,median_percentile_2024,median_percentile_2025,median_percentile_2026,rank,avg_position_last5,median_position_last5,is_rockie
7,126,SNBD,Jacob French,14.0,Pink Powder Club,jacobfrench,0.088384,NaN,NaN,0.090909,0.090909,1.000000,1.0,True
12,165,SNBD,Jarid Palter,14.0,Red,jaridpalter,0.174242,NaN,NaN,0.174242,0.174242,1.750000,2.0,True
8,128,SNBD,Bernie Oegema,14.0,Orange,bernieoegema,0.325758,NaN,NaN,0.272727,0.272727,3.666667,3.0,True
14,213,SNBD,Graham Ramshaw,14.0,Blue,grahamramshaw,0.334918,0.391667,0.266667,0.318182,0.307121,3.600000,3.0,False
9,136,SNBD,Kevin Kilmer-Choi,15.0,Blue,kevinkilmerchoi,0.234946,0.200000,NaN,0.435606,0.341364,7.000000,5.0,False
4,123,SNBD,Barry Pogson,14.0,Big Eggplant Energy,barrypogson,0.462121,NaN,NaN,0.439394,0.439394,4.750000,5.0,True
5,124,SNBD,Steve Crawford,14.0,Black,stevecrawford,0.695800,0.647661,0.777778,0.590909,0.695694,5.600000,6.0,False
10,140,SNBD,Linda Leistner,15.0,Black,lindaleistner,0.537068,0.816667,NaN,0.750000,0.776667,11.200000,9.0,False
1,120,SNBD,Steve Megitt,15.0,Big Eggplant Energy,stevemegitt,0.780303,NaN,NaN,0.780303,0.780303,9.000000,9.0,True
11,144,SNBD,Sheri Ramshaw,15.0,Pink Powder Club,sheriramshaw,0.695570,0.729532,0.894444,0.666667,0.793129,10.400000,9.0,False


# Save outputs!

In [22]:
joined_ski.to_csv("2026/ski_stats_2026.csv", index=False)
joined_snbd.to_csv("2026/snbd_stats_2026.csv", index=False)

# Appendix: looking up racer data

In [19]:
sql = """
SELECT *
FROM RaceResults
WHERE racer_id like '%bernard%'
and discipline = 'SNBD'
;
"""

foo2 = pd.read_sql_query(sql, conn)
foo2

,racer_id,discipline,team,tier,run1,run2,best_time,points,race_id,bib
0,bernardoegema,SNBD,None,NaN,54.53,NaN,54.53,NaN,1,NaN
1,bernardoegema,SNBD,None,NaN,35.09,NaN,35.09,NaN,2,NaN
2,bernardoegema,SNBD,None,NaN,33.28,NaN,33.28,NaN,3,NaN
3,bernardoegema,SNBD,None,NaN,33.27,NaN,33.27,NaN,4,NaN
4,bernardoegema,SNBD,None,NaN,41.44,NaN,41.44,NaN,8,NaN
5,bernardoegema,SNBD,None,NaN,60.22,NaN,60.22,NaN,9,NaN
6,bernardoegema,SNBD,None,NaN,39.09,NaN,39.09,NaN,10,NaN
7,bernardoegema,SNBD,None,NaN,39.04,NaN,39.04,NaN,11,NaN
8,bernardoegema,SNBD,None,NaN,35.71,NaN,35.71,NaN,12,NaN
9,bernardoegema,SNBD,None,NaN,38.73,NaN,38.73,NaN,14,NaN
